# Laboratorio: la temperatura del suelo y la difusividad térmica

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jfbonder/imc/blob/main/notebooks/lab-suelo.ipynb)

Este laboratorio cierra el primer problema conductor de la Parte III: la temperatura del suelo. En el texto planteamos el modelo (la ecuación del calor en el semieje $x>0$ con un forzado periódico en la superficie), lo resolvimos (la **solución periódica** $u_{per} = \bar T + A e^{-x/\delta}\cos(\omega t - x/\delta)$ con $\delta = \sqrt{2D/\omega}$) y dijimos qué se lee de ella: atenuación, retraso, profundidad de penetración. Quedaron pendientes las preguntas 4 y 5 del problema: **dadas mediciones a varias profundidades, ¿se puede estimar la difusividad térmica $D$ del suelo? ¿Se puede comprobar que el modelo es correcto? ¿A qué profundidad hay que enterrar una cañería para que no se congele?** Eso es lo que hacemos acá, con un año de datos horarios.

**Qué vamos a hacer.** Mirar los datos y entender *qué son* (Tarea 1); medir la amplitud y la fase de los ciclos diario y anual en cada profundidad, con la DFT y con mínimos cuadrados con frecuencia fija (Tarea 2); ajustar las rectas de $\ln A$ y de la fase contra la profundidad y obtener **cuatro estimaciones de $\delta$ y de $D$** con su incertidumbre, que no van a coincidir (Tarea 3); ver cuánto de esa discrepancia se debe a que cada dato es un **promedio sobre una capa** (Tarea 4); **simular** el problema completo con el esquema explícito que ya programaron en el laboratorio de EDPs y comparar con las capas, mirando dónde falla el modelo (Tarea 5); repetir la estimación con un solo mes de datos para saber qué precisión es razonable (Tarea 6); y responder la pregunta de la cañería, con las incertidumbres a la vista (Tarea 7). La Tarea 8, opcional, compara qué pasa si se usa el aire como borde.

**Lo que las notas no explican y este notebook sí.** La DFT de un registro con huecos o de un tramo que no contiene períodos enteros, y su alternativa (regresión armónica con frecuencia fija); cómo se propagan las incertidumbres de una regresión a $\delta$ y a $D$ (¡que dependen de $1/\text{pendiente}$ y de su cuadrado!); mínimos cuadrados no lineales con parámetros "molestos" que hay que ajustar aunque no interesen; el promedio de una solución sobre una capa; y los detalles de simular un semiespacio con un dominio finito (profundidad, condición inferior, dato inicial, *spin-up*, paso de tiempo). El texto de las secciones 1 a 6 es la única presentación que van a tener de estos temas: léanlo, no solo las consignas.

**Herramientas disponibles.** `imc.datos.obtener` (los datos), `imc.espectro.amplitudes` (la DFT con la convención de las notas), `imc.espectro.filtrar` (Tarea 8), `imc.estilo`, `scipy.optimize.least_squares` (que se explica en el laboratorio del modelo SIR: si no lo hicieron, lean su Sección 2) y `numpy.linalg.lstsq`. No vamos a volver a construir nada del notebook `09-calor`: allí están la solución periódica y una versión resumida del ajuste de $D$; acá hacemos ese ajuste paso a paso, con incertidumbres, y lo confrontamos con una simulación.

**Cómo se evalúa.** Como siempre: la sección final de **interpretación escrita** (las cinco preguntas del problema conductor, ahora con números). Cada tarea dice qué se espera y trae una celda de verificación. Tiempo estimado: una sesión de 4 h para las Tareas 1 a 6; las Tareas 7 y 8 son la continuación (30 y 40 minutos).

In [ ]:
# Configuración (funciona en Colab y en una copia local del repositorio)
try:
    import imc
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/jfbonder/imc.git"], check=True)
    import imc

import numpy as np
import matplotlib.pyplot as plt
from imc import estilo, datos
from imc.estilo import COLORES
import pandas as pd
from scipy.optimize import least_squares
from imc import espectro
from imc.estilo import CICLO
estilo.activar()

## 1. El modelo, la solución periódica y el dato

### El modelo y su solución periódica

Sea $x \ge 0$ la profundidad y $u(x,t)$ la temperatura del suelo, que suponemos homogéneo con difusividad $D$ (m$^2$/s). Si la temperatura de la superficie es una función conocida $T_s(t)$, el modelo es la ecuación del calor en el semieje:

$$u_t = D\,u_{xx}\ (x>0),\qquad u(0,t) = T_s(t),\qquad u \text{ acotada cuando } x\to\infty .$$

Para un forzado de una sola frecuencia, $T_s = \bar T + A\cos(\omega t + \varphi_s)$, la solución en la que la oscilación tiene la misma frecuencia que el forzado es

$$u_{per}(x,t) = \bar T + A\, e^{-x/\delta}\cos\Bigl(\omega t + \varphi_s - \frac{x}{\delta}\Bigr),\qquad \delta = \sqrt{\frac{2D}{\omega}} .$$

Como la ecuación es lineal, un forzado con varias frecuencias (el ciclo anual, el diario y sus armónicos) da la suma de las soluciones periódicas, **cada una con su propio $\delta_k = \sqrt{2D/\omega_k}$**: las frecuencias altas se apagan más rápido, y por eso la forma de la onda se va redondeando con la profundidad.

**Cómo se lee.** A profundidad $x$, la amplitud es $A(x) = A\,e^{-x/\delta}$ y la fase de la oscilación es $\varphi(x) = \varphi_s - x/\delta$. Conviene trabajar con el **retraso de fase** $\psi(x) = \varphi_s - \varphi(x) = x/\delta$, que crece con la profundidad (el retraso temporal es $\psi/\omega = x/(\delta\omega)$). Las dos funciones que vamos a medir son entonces *rectas en $x$*:

$$\ln A(x) = \ln A(0) - \frac{x}{\delta},\qquad \psi(x) = \psi(0) + \frac{x}{\delta}\quad(\text{con }\psi(0)=0\text{ si se mide respecto de la superficie}).$$

(El texto escribe que $\varphi(x)$ tiene pendiente $1/\delta$; con la convención $\cos(\omega t + \varphi)$ de la Parte II la pendiente de $\varphi$ es $-1/\delta$, y la de $\psi = -\varphi$ es $+1/\delta$. Acá usamos $\psi$.)

**Los números que hay que tener en la cabeza** (con el valor típico $D = 5\times10^{-7}$ m$^2$/s): $\delta_{\text{diario}} = \sqrt{2D/\omega_d} \approx 12$ cm y $\delta_{\text{anual}} \approx 2.2$ m; el cociente es $\sqrt{365} \approx 19$. A profundidad $\delta$ la amplitud es el 37 % de la de la superficie y el retraso es $1/\omega$: unas 4 horas para el ciclo diario y unos 2 meses para el anual.

### El dato: qué es y qué no es

El archivo `suelo_horaria.csv` (ver `datos/README.md`) tiene el año 2023 completo, cada hora (8760 filas, sin huecos, hora local), en un punto de Buenos Aires (Aeroparque): la temperatura del aire (`t_aire`) y la de cuatro capas de suelo, $0$–$7$, $7$–$28$, $28$–$100$ y $100$–$255$ cm.

**Advertencia de modelado, y es parte de la lección.** El enunciado del ejercicio habla de "registros de temperatura a varias profundidades". Estos datos no son eso: son de un **reanálisis** (ERA5-Land: un modelo de superficie terrestre que asimila observaciones) y cada columna es la temperatura **promedio de una capa** de espesor entre $7$ y $155$ cm, no la de un sensor enterrado a una profundidad dada. Tres consecuencias que van a aparecer una y otra vez:

1. Hay que asignarle a cada capa una **profundidad representativa**. Usamos el punto medio: $x = 3.5,\ 17.5,\ 64$ y $177.5$ cm. Es una aproximación; en la Tarea 4 vemos cuándo es buena y cuándo no.
2. El dato *es la salida de un modelo* de suelo, con sus propias hipótesis (suelo estratificado, humedad variable, un forzado atmosférico completo). Que nuestro modelo simple ajuste o no ajuste dice algo sobre la *consistencia* de los dos modelos, no sobre un suelo real. Con sensores enterrados a $5$, $10$, $20$, $50$ y $100$ cm el tratamiento sería el mismo, pero los promedios de capa desaparecerían.
3. No tenemos la temperatura en $x=0$. La capa superficial ($0$–$7$ cm) es lo más cercano: no es el aire (ver Sección 5) ni la superficie misma.

También hay que recordar que estamos en el hemisferio sur: enero es verano y julio es invierno.

In [ ]:
capas = ["t_suelo_0_7", "t_suelo_7_28", "t_suelo_28_100", "t_suelo_100_255"]
nombres = ["0–7 cm", "7–28 cm", "28–100 cm", "100–255 cm"]
bordes = np.array([0.0, 0.07, 0.28, 1.00, 2.55])        # límites de las capas (m)
x_m = 0.5 * (bordes[:-1] + bordes[1:])                  # profundidad representativa: punto medio (m)
omega_d = 2 * np.pi / 86400                             # ciclo diario (rad/s)
omega_a = 2 * np.pi / (365 * 86400)                     # ciclo anual (rad/s)
D_tipico = 5e-7                                         # m^2/s
delta_de = lambda D, omega: np.sqrt(2 * D / omega)      # profundidad de penetración
D_de = lambda delta, omega: delta ** 2 * omega / 2      # difusividad a partir de delta
print("profundidades representativas (m):", x_m)
print(f"con D = {D_tipico:.0e}:  delta diario = {100 * delta_de(D_tipico, omega_d):.1f} cm,  delta anual = {delta_de(D_tipico, omega_a):.2f} m")

### Tarea 1: Cargar, graficar y describir

Cargá los datos con `pd.read_csv(datos.obtener("suelo_horaria.csv"), parse_dates=["fecha_hora"])` y construí: `N` (cantidad de filas), `t` (tiempo en **segundos** desde la primera fila, array de largo `N`), `aire` (array de la temperatura del aire) e `Y` (array `(N, 4)` con las cuatro capas, en el orden de `capas`). Las cantidades físicas están en unidades SI (metros, segundos): así $D$ sale en m$^2$/s.

Después:

1. Verificá que no hay huecos (que el paso es siempre de una hora) y armá una tabla con la media, el desvío, el mínimo y el máximo del aire y de cada capa.
2. Graficá, en una figura de cuatro paneles: **(a)** una semana de enero (del 9 al 16) hora por hora, con el aire y las cuatro capas; **(b)** una semana de julio (del 10 al 17); **(c)** el año completo en medias diarias; **(d)** el **ciclo diario medio**: la temperatura media a cada hora del día (0 a 23) para cada serie.
3. De (d) sacá, para cada serie, la hora del día en que ocurre el máximo del ciclo medio y la amplitud pico a pico (máximo menos mínimo).

**Qué se espera.** En (a) el ciclo diario se atenúa y se retrasa capa a capa (a $28$–$100$ cm ya casi no se ve); en (b) es mucho más chico que en (a): la amplitud diaria depende de la estación. En (c) el ciclo anual también se atenúa y se retrasa, pero mucho menos. En (d) el máximo de la capa superficial ocurre alrededor de las 15 h (como el del aire, pero con más amplitud) y el de la capa $7$–$28$ cm cuatro horas después; en las capas más profundas el ciclo medio es de centésimas de grado y la "hora del máximo" no significa nada. Anotá qué observás sobre la capa superficial respecto del aire (¿oscila más o menos? ¿por qué podría ser?): es lo que hace que el aire no sea un buen borde (Sección 5).

In [ ]:
suelo = ...    # TODO: pd.read_csv(datos.obtener("suelo_horaria.csv"), parse_dates=["fecha_hora"])
N = ...        # TODO
t = ...        # TODO: segundos desde la primera fila
aire = ...     # TODO
Y = ...        # TODO: array (N, 4) con las capas

# TODO: verificar que el paso es siempre de 1 hora; tabla con media, desvío, mínimo y máximo
# TODO: figura de cuatro paneles (semana de enero, semana de julio, año en medias diarias, ciclo diario medio)
# TODO: hora del máximo y amplitud pico a pico del ciclo diario medio de cada serie

In [ ]:
# Verificación
assert N == 8760 and Y.shape == (8760, 4) and aire.shape == (8760,) and t.shape == (8760,)
assert np.allclose(np.diff(t), 3600) and not np.isnan(Y).any(), "el paso debe ser de 3600 s y no debe haber NaN"
assert 18 < Y.mean() < 20 and Y[:, 0].std() > Y[:, 3].std(), "la capa superficial oscila más que la profunda"
print("datos: OK")

## 2. Amplitud y fase de los ciclos diario y anual

### Con la DFT

Un año tiene $N = 8760$ muestras horarias, es decir $T = 8760$ h $= 365$ días exactos. La DFT (Parte II) da las frecuencias $\xi_k = k/T$: el **ciclo anual** es exactamente $k = 1$ y el **ciclo diario** es $k = 365$ (24 h $= T/365$). Es la razón por la que estos datos son cómodos: contienen un número *entero* de períodos de cada ciclo, y entonces no hay pérdida espectral (*leakage*). Con la convención de las notas, la componente $k$ de la serie $x_n$ es $A_k\cos(2\pi k n/N + \varphi_k)$ con

$$A_k = \frac{2|X_k|}{N},\qquad \varphi_k = \arg X_k,\qquad X_k = \sum_{n=0}^{N-1} x_n\,e^{-2\pi i kn/N}\quad(0<k<N/2).$$

La fase se mide con el origen de tiempos en la primera fila (el 1/1 a las 00:00): es un origen común a todas las series, y eso es lo único que importa, porque de la fase solo usamos **diferencias entre profundidades**. `imc.espectro.amplitudes(x, dt)` devuelve `(xi, A, phi)` con esta convención.

### Con mínimos cuadrados y frecuencia fija

La alternativa es plantear la **regresión armónica**: ajustar a los datos $x(t_n)$ el modelo

$$x(t) \approx c_0 + \sum_{j} \bigl[a_j\cos(\omega_j t) + b_j\sin(\omega_j t)\bigr]$$

con las frecuencias $\omega_j$ *fijas* (las que nos interesan). El modelo es lineal en $c_0, a_j, b_j$, así que es la regresión múltiple de Estadística: se arma la matriz de diseño $M$ (una columna de unos, y un par coseno/seno por frecuencia), se resuelve `np.linalg.lstsq(M, x)` y la amplitud y la fase salen de $a_j\cos\omega t + b_j\sin\omega t = A_j\cos(\omega t + \varphi_j)$, es decir

$$A_j = \sqrt{a_j^2 + b_j^2},\qquad \varphi_j = \operatorname{atan2}(-b_j,\ a_j).$$

**Para qué sirve si ya tenemos la DFT.** En un año completo y sin huecos, las columnas de $M$ son **ortogonales** (los senos y cosenos de períodos enteros lo son) y la solución de mínimos cuadrados coincide *exactamente* con la DFT: en esta parte del laboratorio dan lo mismo hasta el redondeo. La regresión es más robusta cuando la DFT no se puede usar bien:

* **Con huecos.** La DFT necesita muestras uniformes: hay que rellenar los huecos (con ceros, con la media, interpolando) y todo relleno le quita señal al ciclo (si falta el 20 % de las horas y las rellenás con la media, la amplitud del ciclo diario se subestima en un 20 %). La regresión usa solo las filas que existen.
* **Con tramos cortos o que no contienen períodos enteros**, como un mes de datos (Tarea 6). Se pueden agregar a $M$ **columnas de tendencia** (un polinomio de grado bajo en el tiempo) para que la variación lenta, como el ciclo anual visto en un mes, no contamine el ciclo diario, y **armónicos** ($2\omega_j$) para que la forma no sinusoidal de la onda tampoco lo haga.
* **Cuando se quiere la incertidumbre** de $a_j, b_j$: la regresión da la matriz de covarianza $\hat\sigma^2(M^\top M)^{-1}$.

**Qué se espera de la Tarea 2.** Que las dos tablas (DFT y mínimos cuadrados) coincidan hasta $10^{-9}$; y que con el registro con huecos que armamos, la DFT con interpolación subestime la amplitud diaria (del orden del 15–20 %, la fracción de horas que faltan) y la regresión no (error de unos pocos por ciento).

In [ ]:
# Un registro con huecos: borramos el 19 % de las horas en tramos de 6 a 48 horas y una semana entera (semilla fija)
rng = np.random.default_rng(2023)
mask = np.ones(N, bool)
for _ in range(60):
    i0 = rng.integers(0, N - 48)
    mask[i0:i0 + rng.integers(6, 48)] = False
mask[3000:3168] = False
print(f"horas que faltan: {100 * (1 - mask.mean()):.1f} %")

### Tarea 2: Amplitud y fase de cada ciclo, por dos métodos

Escribí:

* `dft_ciclo(y, k)`: devuelve `(A, phi)` del armónico `k` de la serie `y`, con la convención de arriba.
* `ls_ciclos(y, t, omegas, grado=0)`: regresión armónica por mínimos cuadrados con las frecuencias `omegas` (rad/s) fijas; la matriz de diseño lleva una columna por cada potencia $\tilde t^j$, $j = 0, \dots,$ `grado` (con $\tilde t$ el tiempo normalizado a $[-1/2, 1/2]$), y un par $\cos(\omega t)$, $\sin(\omega t)$ por frecuencia. Devuelve una lista de pares `(A, phi)`, uno por frecuencia.

**(a)** Armá el DataFrame `tab_dft` (filas: aire y las cuatro capas, con `nombres_s = ["aire"] + nombres`; columnas `A_anual`, `phi_anual`, `A_diaria`, `phi_diaria`) con la DFT ($k = 1$ y $k = 365$), y `tab_ls` con lo mismo por mínimos cuadrados (con `omegas = [omega_a, omega_d]`). Imprimí la tabla y la diferencia máxima entre las dos.

**(b)** *El registro con huecos.* La celda de arriba de esta tarea arma una máscara `mask` que borra el 19 % de las horas en tramos de 6 a 48 horas (y una semana entera). Para las capas superficiales ($0$–$7$ y $7$–$28$ cm) estimá el **ciclo diario** de tres maneras: (i) la referencia del año completo (`tab_dft`); (ii) DFT del registro con los huecos rellenados por interpolación lineal (`np.interp`); (iii) mínimos cuadrados usando solo las filas con `mask` (agregá los armónicos $2\omega_a$ y $2\omega_d$ para reducir la contaminación). Mostrá el error relativo de la amplitud y el error de la fase de (ii) y (iii).

**Qué se espera.** (a): diferencias del orden de $10^{-12}$. Valores de referencia para chequear: la amplitud anual del aire es de unos $7$ °C y la de la capa superficial de unos $8$ °C; la amplitud diaria de la capa superficial es de unos $4$ °C y baja a menos de $1$ °C en $7$–$28$ cm y a $0.03$ °C en $28$–$100$ cm. (b): la DFT con interpolación pierde entre el 15 y el 20 % de amplitud; mínimos cuadrados, unos pocos por ciento (o menos).

In [ ]:
def dft_ciclo(y, k):
    """Amplitud y fase del armónico k de la serie y (convención A cos(2 pi k n/N + phi))."""
    # TODO
    pass

def ls_ciclos(y, t, omegas, grado=0):
    """Regresión armónica con frecuencias fijas. Devuelve [(A_1, phi_1), (A_2, phi_2), ...]."""
    # TODO: matriz de diseño (tendencia de grado `grado` + cos/sin de cada omega), lstsq, y (A, phi) = (hypot(a, b), atan2(-b, a))
    pass

series = [aire] + [Y[:, i] for i in range(4)]
nombres_s = ["aire"] + nombres
columnas = ["A_anual", "phi_anual", "A_diaria", "phi_diaria"]
tab_dft = ...   # TODO: DataFrame 5 x 4 con la DFT (k = 1 y k = 365)
tab_ls = ...    # TODO: lo mismo por mínimos cuadrados con omegas = [omega_a, omega_d]
# TODO: imprimir las tablas y la diferencia máxima

# (b) registro con huecos
# TODO: comparar, para las capas 0 y 1, referencia / DFT con interpolación / mínimos cuadrados con máscara

In [ ]:
# Verificación
assert np.abs(tab_dft.values - tab_ls.values).max() < 1e-8, "DFT y mínimos cuadrados deberían coincidir en un año completo"
assert abs(tab_dft.loc["aire", "A_anual"] - 6.9) < 0.3 and abs(tab_dft.loc["0–7 cm", "A_diaria"] - 4.0) < 0.2
assert tab_dft.loc["28–100 cm", "A_diaria"] < 0.05
assert abs(dft_ciclo(3 * np.cos(2 * np.pi * 5 * np.arange(100) / 100 + 0.7), 5)[0] - 3) < 1e-9, "dft_ciclo: A = 2|X_k|/N"
print("amplitud y fase: OK")

## 3. Las rectas de $\ln A$ y de $\psi$ contra $x$: cuatro estimaciones de $\delta$

### Qué se ajusta

Con $A_k$ y $\varphi_k$ de cada capa, el modelo predice dos rectas en $x$ con la *misma* pendiente en valor absoluto:

$$\ln A(x) = \ln A(0) - \frac{x}{\delta},\qquad \psi(x) = \psi(0) + \frac{x}{\delta}.$$

De cada recta sale un $\delta = 1/|\text{pendiente}|$, y de $\delta$ sale $D = \delta^2\omega/2$: **dos estimaciones por ciclo, cuatro en total** (amplitud y fase, diario y anual). Como $\omega_d$ y $\omega_a$ son conocidos, la incertidumbre de $D$ viene solo de la de la pendiente.

**La fase, con cuidado.** La fase de cada capa es un ángulo: está definida módulo $2\pi$. Para que $\psi(x)$ sea una función creíble de $x$ hay que **desenrollarla**: como el retraso entre capas consecutivas es menor que $\pi$, alcanza con sumar a cada diferencia $\varphi_{i+1} - \varphi_i$ el múltiplo de $2\pi$ que la deja en $(-\pi, \pi]$ y acumular. Como no tenemos la superficie, medimos el retraso respecto de la primera capa, $\psi_i - \psi_1$; la ordenada al origen de la recta absorbe lo que falta.

**Qué capas usar.** Para el ciclo anual, las cuatro. Para el diario, las dos superficiales: en la tercera ($28$–$100$ cm) la amplitud diaria es de $0.03$ °C, la señal es apenas más grande que el error de cuantización de los datos (cada dato tiene un decimal), y su "retraso" es dudoso (la capa mide $72$ cm, seis veces $\delta_{\text{diario}}$: Tarea 4). Con dos puntos la recta pasa exacta y no hay grados de libertad para estimar una incertidumbre; para ese caso proponemos otro camino (más abajo).

### Incertidumbre de la regresión, y su propagación

De Estadística: para $n$ puntos $(x_i, y_i)$ y la recta $y = a + bx$ por mínimos cuadrados, con residuos $e_i$,

$$\hat\sigma^2 = \frac{\sum e_i^2}{n-2},\qquad \operatorname{se}(b) = \frac{\hat\sigma}{\sqrt{S_{xx}}},\quad S_{xx} = \sum (x_i - \bar x)^2,\qquad \operatorname{se}(a) = \hat\sigma\sqrt{\frac1n + \frac{\bar x^2}{S_{xx}}} .$$

Con $n = 4$ hay solo **2 grados de libertad**: el intervalo de confianza del 95 % usa el cuantil $t_{2,\,0.975} = 4.3$, no $1.96$. Y $\delta$ y $D$ *no son lineales* en la pendiente: $\delta = 1/|b|$, $D = \delta^2\omega/2 = \omega/(2b^2)$. Con la aproximación de primer orden (delta method),

$$\operatorname{se}(\delta) \approx \frac{\operatorname{se}(b)}{b^2},\qquad \frac{\operatorname{se}(D)}{D} \approx \frac{2\operatorname{se}(b)}{|b|} = \frac{2\operatorname{se}(\delta)}{\delta} .$$

Una pendiente con 10 % de error relativo da un $D$ con 20 %. Además, si la pendiente es chica respecto de su error, $\delta$ y $D$ explotan.

**Para el ciclo diario: remuestreo de días.** Hay una incertidumbre que la regresión de dos puntos no ve, pero que se puede medir: cuánto cambiaría la amplitud y la fase del ciclo diario de cada capa si tuviéramos *otros* días. Para eso calculamos la amplitud compleja $C_d = \frac{2}{24}\sum_h y_{d,h}\,e^{-i\omega_d t_h}$ **de cada día** $d$ (el promedio de los 365 $C_d$ es exactamente la DFT con $k=365$) y hacemos un **bootstrap por bloques**: se sortean con reposición bloques de 10 días consecutivos (los días vecinos se parecen, así que remuestrear días sueltos subestimaría la variabilidad), se promedian los $C_d$ elegidos, se calculan $A$ y $\varphi$ de las dos capas y de ahí el $\delta$ y el $D$; con 1000 remuestreos, el desvío de esos $D$ es una medida de la incertidumbre. No es una incertidumbre "de regresión" sino de *muestreo temporal*: es la que refleja que enero y julio no son iguales.

**Qué se espera.** Las cuatro estimaciones de $D$ están entre $3\times10^{-7}$ y $2\times10^{-6}$ m$^2$/s: el ciclo diario da valores del orden de $3$–$5\times10^{-7}$ (por amplitud y por fase, parecidos entre sí y al valor típico); el anual da por fase un valor de unos $3\times10^{-7}$ y por amplitud un valor **cinco veces mayor**. Las incertidumbres estadísticas que calcules van a ser chicas (de unos pocos por ciento, salvo la del anual por amplitud, cerca del $20\,\%$): **las diferencias entre las estimaciones son mucho mayores que sus incertidumbres**. Eso es lo que distingue un error de medición (que la incertidumbre describe) de un *error de modelo* (que no describe): si las cuatro estimaciones fueran del mismo $D$, tendrían que coincidir dentro de sus incertidumbres, y no lo hacen. Que dos estimaciones del mismo ciclo difieran por un factor 5 no es un error de programación: es el hallazgo del laboratorio. La pregunta 3 del enunciado ("¿coinciden? ¿deberían?") se responde con esta tarea y con la siguiente.

### Tarea 3: Rectas, cuatro estimaciones de $\delta$ y $D$, incertidumbres

Escribí:

* `recta(x, y)`: ajuste $y = a + bx$; devuelve `(a, b, se_a, se_b)` con los errores estándar de arriba (`np.nan` si hay solo dos puntos).
* `retraso_de_fase(phi)`: desenrolla las fases de capas consecutivas y devuelve $\psi_i - \psi_1$ (creciente con la profundidad).
* `estimar(x, y, omega, tipo)`: con `tipo="amp"` (y = $\ln A$, pendiente $-1/\delta$) o `tipo="fase"` (y = $\psi$, pendiente $+1/\delta$), devuelve `(delta, se_delta, D, se_D)` propagando la incertidumbre como arriba.

Con eso llená el diccionario `est` con las cuatro estimaciones: claves `("anual", "amp")`, `("anual", "fase")`, `("diario", "amp")`, `("diario", "fase")` (anual con las cuatro capas, diario con las dos superficiales). Para el ciclo diario, calculá la incertidumbre con el **bootstrap por bloques** descripto arriba (`Cd`: array `(365, 4)` de amplitudes complejas por día) y guardala en `se_D`.

Imprimí una tabla con $\delta$, $D$ y sus incertidumbres, y graficá, para cada ciclo, en un mismo panel y contra $x$: la **atenuación** $-(\ln A - \ln A_1)$ y el **retraso** $\psi - \psi_1$, con sus dos rectas ajustadas. Si el modelo valiera, las dos curvas coincidirían.

Como extra, calculá el $D$ diario con las *tres* capas superficiales (regresión de tres puntos, con su error estándar) y comentá qué le pasa a la estimación por fase.

**Qué se espera.** Un gráfico (dos paneles, ciclo anual y ciclo diario) en el que atenuación y retraso son casi rectas pero con pendientes distintas: en el anual, la atenuación es mucho más lenta que el retraso. Valores de $D$ (orden de magnitud): diario por amplitud y por fase entre $3$ y $5\times10^{-7}$; anual por fase $\approx 3\times10^{-7}$, por amplitud $\approx 1.6\times10^{-6}$. Incertidumbres estadísticas chicas (unos pocos por ciento, salvo la del anual por amplitud, cerca del $20\,\%$): las diferencias entre las estimaciones son mucho mayores que ellas.

In [ ]:
def recta(x, y):
    """Ajuste y = a + b x. Devuelve (a, b, se_a, se_b)."""
    # TODO
    pass

def retraso_de_fase(phi):
    """Retraso de fase respecto de la primera capa, desenrollando las diferencias entre capas consecutivas."""
    # TODO
    pass

def estimar(x, y, omega, tipo):
    """tipo = "amp" (y = ln A, pendiente -1/delta) o "fase" (y = psi, pendiente +1/delta). Devuelve (delta, se_delta, D, se_D)."""
    # TODO
    pass

# amplitudes complejas por día y por capa (365 x 4): Cd[d, i] = (2/24) sum_h y[d, h, i] exp(-i omega_d t_h)
E = np.exp(-1j * omega_d * np.arange(24) * 3600)
Cd = (2 / 24) * (Y.reshape(365, 24, 4) * E[None, :, None]).sum(axis=1)

est = {}
# TODO: est[("anual", "amp")], est[("anual", "fase")]  (cuatro capas)
# TODO: est[("diario", "amp")], est[("diario", "fase")]  (dos capas; bootstrap por bloques para se_D)
# TODO: tabla y figura (atenuación y retraso contra x, con las dos rectas, para cada ciclo)
# TODO (extra): D diario con las tres capas superficiales

In [ ]:
# Verificación
D_ad, D_fd = est[("diario", "amp")][2], est[("diario", "fase")][2]
D_aa, D_fa = est[("anual", "amp")][2], est[("anual", "fase")][2]
assert 2.5e-7 < D_ad < 4.5e-7, f"diario por amplitud {D_ad:.2e}: se esperaba ~3.3e-7"
assert 3.5e-7 < D_fd < 5.5e-7, f"diario por fase {D_fd:.2e}: se esperaba ~4.5e-7"
assert 2.5e-7 < D_fa < 3.8e-7, f"anual por fase {D_fa:.2e}: se esperaba ~3.1e-7"
assert 1.2e-6 < D_aa < 2.2e-6, f"anual por amplitud {D_aa:.2e}: se esperaba ~1.6e-6"
assert np.all(np.diff(retraso_de_fase(tab_dft.phi_anual.values[1:])) > 0), "el retraso debe crecer con la profundidad"
print("cuatro estimaciones de D: OK")

## 4. Cada dato es un promedio de capa: qué corrige y qué no

### El promedio de la solución periódica

Lo que mide (o mejor: lo que informa) el dato $k$ no es $u(x_k, t)$ sino el promedio sobre la capa $[a, b]$ de espesor $h = b - a$:

$$\bar u_{[a,b]}(t) = \frac1h\int_a^b u(x,t)\,dx .$$

Escribamos la solución periódica con notación compleja, $u_{per} = \bar T + \operatorname{Re}\bigl[\,A e^{i\varphi_s}\,e^{i\omega t}\,e^{-zx}\bigr]$ con $z = (1+i)/\delta$. Como el promedio es lineal y $e^{i\omega t}$ no depende de $x$,

$$\bar u_{[a,b]}(t) = \bar T + \operatorname{Re}\bigl[\,A e^{i\varphi_s}\; g(\delta; a, b)\; e^{i\omega t}\bigr],\qquad g(\delta; a, b) = \frac1h\int_a^b e^{-zx}\,dx = \frac{e^{-za} - e^{-zb}}{z\,h}.$$

La capa se comporta entonces como un punto con **amplitud** $A|g|$ y **fase** $\varphi_s + \arg g$: el promedio cambia la amplitud por el factor $|g|$ y adelanta o atrasa la fase en $\arg g$, con $g$ una función *conocida* de $\delta$. Para un punto a profundidad $x$ sería $g = e^{-zx}$. Escribiendo $e^{-za} - e^{-zb} = 2e^{-zx_m}\sinh(zh/2)$ con $x_m = (a+b)/2$, queda

$$g(\delta; a, b) = e^{-z x_m}\;\frac{\sinh w}{w},\qquad w = \frac{zh}{2} = (1+i)\frac{h}{2\delta}:$$

**la capa es el punto medio multiplicado por la corrección $\sinh w/w$**, que solo depende de $h/\delta$. Para una capa fina ($h \ll \delta$), $\sinh w/w \approx 1 + w^2/6 = 1 + i\,h^2/(12\delta^2)$: el módulo cambia en $O((h/\delta)^4)$ y la fase en $h^2/(12\delta^2)$, es decir, el promedio está *un poco menos retrasado* que el punto medio. Para una capa gruesa ($h \gtrsim \delta$), $\sinh w/w$ crece como $e^{h/2\delta}$: el promedio queda dominado por el borde *superior* de la capa, donde la oscilación es más grande.

**Atención con la intuición.** Es tentador pensar que "promediar una capa gruesa reduce la amplitud" (mezcla partes con fase distinta). Es al revés: la amplitud de la capa es **mayor** que la del punto medio (porque $e^{-x/\delta}$ es convexa, el promedio en una capa pesa más lo de arriba) y el retraso es **menor**. Y el efecto solo importa cuando $h$ es comparable con $\delta$ del ciclo considerado: enorme para el ciclo diario en la capa $28$–$100$ cm ($h = 72$ cm $\approx 6\delta_d$), y despreciable para el ciclo anual en todas las capas ($h_{max} = 1.55$ m $< \delta_a = 2.2$ m). La Tarea 4 pide calcular esos factores.

### Ajuste con la fórmula integrada

Con el promedio como modelo, ya no hay dos rectas separadas sino **un modelo con tres parámetros** que explica a la vez las amplitudes y las fases de todas las capas de un ciclo. Sea $C_k = A_k e^{i\varphi_k}$ la amplitud compleja medida en la capa $k$. El modelo es

$$C_k \approx C_s\,g(\delta; a_k, b_k),\qquad C_s = e^{\ln A_0}\,e^{-i\psi_0}$$

con parámetros $p = (\ln A_0, \psi_0, \delta)$: la amplitud $A_0$ y la fase $-\psi_0$ **del forzado en $x = 0$** (no las conocemos: la capa superficial es ella misma un promedio) y la profundidad de penetración $\delta$. Los residuos son adimensionales: $\ln|C_k| - \ln|C_s g_k|$ (error relativo de la amplitud) y $\arg(C_k/(C_s g_k))$ (error de fase, en $(-\pi,\pi]$), y se minimiza la suma de cuadrados de todos ellos con `least_squares`.

**Lo que hay que entender de este ajuste** (los mínimos cuadrados no lineales se explican en la Sección 2 del laboratorio SIR; acá van solo las novedades):

* **Parámetros molestos.** $A_0$ y $\psi_0$ no interesan pero hay que ajustarlos: sin ellos, tendríamos que suponer que la superficie tiene la amplitud y la fase de la capa $0$–$7$ cm. Parametrizamos con $\ln A_0$ para que la amplitud sea positiva sin cotas.
* **Amplitud y fase a la vez.** Las dos rectas de la Tarea 3 daban $\delta$ distintos; este ajuste da **un** $\delta$, un compromiso entre las dos informaciones. No es "mejor": si el modelo no es correcto, el compromiso es solo una manera de promediar dos estimaciones inconsistentes, y los residuos (que hay que mirar) lo delatan.
* **Grados de libertad.** Cada capa aporta dos datos (amplitud y fase). Con 4 capas hay 8 datos y 3 parámetros (5 grados de libertad); con las 2 superficiales del ciclo diario, 4 datos y 3 parámetros: **1 solo grado de libertad**, casi identificable. La covarianza de la estimación es la de las notas del SIR, $\hat\sigma^2(J^\top J)^{-1}$ con $\hat\sigma^2 = \mathrm{RSS}/(n-3)$; con un solo grado de libertad la incertidumbre es orientativa.
* **Valores iniciales.** El $\delta$ inicial puede estar a un orden de magnitud del verdadero (¡el ciclo diario y el anual difieren por 19!), y la función de costo tiene mínimos locales. Usá varios arranques con $\delta_0$ repartidos en escala logarítmica (`np.geomspace(0.03, 5, 8)`) y quedate con el de menor costo. Cotas: $\delta \in [0.01, 20]$ m.
* **La fase módulo $2\pi$** en los residuos: `np.angle(C / modelo)` ya devuelve un ángulo en $(-\pi, \pi]$, no hace falta desenrollar.

**Qué se espera.** (i) Los factores de corrección $|g|/|e^{-zx_m}|$ y $\arg g - \arg e^{-zx_m}$ para las cuatro capas y los dos ciclos, con $\delta$ de $D = 5\times10^{-7}$: casi $1$ y casi $0$ para el ciclo anual en todas las capas y para el diario en la capa superficial; en la capa $7$–$28$ cm el ciclo diario tiene un factor $1.01$ pero un adelanto de fase de una hora; y a partir de la tercera capa el factor es de unos $2.5$ con un corrimiento de fase de más de $2$ rad. (ii) Un $D$ anual **entre las dos estimaciones de la Tarea 3** (de un orden de $5$–$6\times10^{-7}$), prácticamente igual con y sin promedio de capa. (iii) Para el ciclo diario con tres capas, el modelo puntual da un ajuste malo y un $D$ absurdo, y el modelo con promedio ajusta con residuos chicos y da un $D$ del orden de $4\times10^{-7}$. Así que el promedio de capas **no explica** la discrepancia del ciclo anual; sí permite usar la tercera capa en el ciclo diario.

### Tarea 4: Corrección por promedio de capa y ajuste no lineal de $\delta$

Escribí:

* `ganancia(delta, a, b)`: la función $g(\delta; a, b)$ de arriba (compleja). Verificala contra la integral numérica (promedio de `np.exp(-(1+1j)*x/delta)` en una grilla fina de $[a,b]$).
* `ajustar_capas(C, idx, omega, modo="capa")`: ajusta $(\ln A_0, \psi_0, \delta)$ a las amplitudes complejas `C[idx]` (las capas de índices `idx`) con `least_squares` y varios arranques; con `modo="punto"` usa $g = e^{-zx_m}$ (profundidad = punto medio) en lugar del promedio. Devuelve un diccionario con `delta`, `se_delta` (de $\hat\sigma^2(J^\top J)^{-1}$; `np.nan` si no hay grados de libertad), `D` (con `omega`), `se_D` (propagando como en la Tarea 3), `A0`, `psi0` y `rss` (suma de cuadrados de los residuos).

Después: **(a)** una tabla de los factores $|g|/|g_{punto}|$ y de $\arg g - \arg g_{punto}$ (en horas o días) para cada capa y cada ciclo (con $\delta$ de $D = 5\times10^{-7}$); **(b)** los ajustes: anual con las 4 capas y diario con 3 y con 2 capas, en los dos modos, con una tabla de $\delta$, $D$ y RSS; **(c)** una figura con tres paneles: la amplitud diaria contra $x$ (datos, y ajuste puntual y de capa evaluados en cada capa), lo mismo para la anual, y el factor $|\sinh w/w|$ contra $h/\delta$ con las capas de los dos ciclos marcadas.

Guardá en `D_capa_a` y `D_capa_d` los $D$ del ajuste con promedio de capa del ciclo anual (4 capas) y del diario (3 capas).

**Qué se espera.** Ver la lista de "qué se espera" de arriba. Para el ciclo anual, $D \approx 6\times10^{-7}$; para el diario con tres capas, $D \approx 4\times10^{-7}$ (con promedio) contra $\approx 10^{-6}$ (puntual). Con dos capas el diario da $\approx 3.8\times10^{-7}$ (puntual) y $\approx 2.9\times10^{-7}$ (de capa): la corrección de la capa $7$–$28$ cm ($h\approx 1.8\,\delta_d$) ya se nota en la fase. En todos los casos, mirá también el error estándar de $D$: es de decenas de por ciento (hay pocos grados de libertad).

In [ ]:
def ganancia(delta, a, b):
    """Promedio sobre [a, b] de exp(-(1+i) x / delta)."""
    # TODO
    pass

def ajustar_capas(C, idx, omega, modo="capa"):
    """Ajusta (ln A0, psi0, delta) a las amplitudes complejas C[idx]. Devuelve un dict con
    delta, se_delta, D, se_D, A0, psi0, rss."""
    # TODO: residuos = [ln|C| - ln|modelo|, angle(C/modelo)], modelo = exp(lnA0 - i psi0) * g(delta)
    #       varios arranques (delta0 en geomspace), cotas, covarianza s2 (J^T J)^-1 con s2 = rss/(n - 3)
    pass

C_a = tab_dft.A_anual.values[1:] * np.exp(1j * tab_dft.phi_anual.values[1:])      # amplitudes complejas medidas (4 capas)
C_d = tab_dft.A_diaria.values[1:] * np.exp(1j * tab_dft.phi_diaria.values[1:])

# TODO (a): factores de corrección por capa y por ciclo
# TODO (b): ajustes: anual (4 capas), diario (3 capas y 2 capas), modos "punto" y "capa"
# TODO (c): figura de tres paneles
D_capa_a = ...
D_capa_d = ...

In [ ]:
# Verificación
xs = 0.07 + (np.arange(20000) + 0.5) * 0.21 / 20000
assert abs(ganancia(0.117, 0.07, 0.28) - np.mean(np.exp(-(1 + 1j) * xs / 0.117))) < 1e-6, "ganancia: no coincide con la integral numérica"
assert abs(ganancia(1.0, 0.0, 1e-9) - 1) < 1e-6, "una capa de espesor nulo en x = 0 debe dar g = 1"
assert 4e-7 < D_capa_a < 8e-7, f"D anual con promedio de capa {D_capa_a:.2e}: se esperaba ~6e-7"
assert 3e-7 < D_capa_d < 5e-7, f"D diario (3 capas) con promedio de capa {D_capa_d:.2e}: se esperaba ~4e-7"
print("corrección por capas: OK")

## 5. Simular el problema completo y ver dónde falla

Las estimaciones de las Tareas 3 y 4 usan solo una amplitud y una fase por ciclo. La simulación pone a prueba el modelo con **todo** el registro: toda la señal de la superficie (el ciclo diario y el anual, sus armónicos, las olas de calor, la lluvia...) entra como condición de borde, la ecuación del calor la propaga hacia abajo, y comparamos con lo que dice cada capa. Si el modelo es adecuado, la diferencia es "ruido"; si no, tiene estructura.

### El problema numérico

Un semiespacio no se puede simular: hay que cortarlo. Resolvemos

$$u_t = D\,u_{xx}\ \ (0<x<L,\ t>0),\qquad u(0,t) = T_s(t),\qquad u(L,t) = T_L\ \ (\text{o } u_x(L,t)=0),\qquad u(x,0) = u_0(x),$$

con el **esquema explícito** del Ejercicio de la ecuación del calor (laboratorio de EDPs): con $x_j = j\Delta x$, $t_n = n\Delta t$,

$$u_j^{n+1} = u_j^n + r\,(u_{j+1}^n - 2u_j^n + u_{j-1}^n),\qquad r = \frac{D\,\Delta t}{\Delta x^2},$$

que es estable si y solo si $r \le 1/2$. Decisiones que hay que tomar (y que son parte del modelado, no detalles):

* **Profundidad $L$.** La solución periódica anual decae como $e^{-x/\delta_a}$ con $\delta_a \approx 2.2$ m: a $L = 10$ m queda el 1 % de la amplitud de la superficie. Lo que importa es que el borde inferior no afecte a las capas que comparamos (hasta $2.55$ m): se verifica variando $L$ y la condición inferior. Con $L$ chico, el fondo "ancla" la temperatura y las capas profundas se aplanan.
* **Condición inferior.** $u(L) = T_L$ con $T_L$ la temperatura media anual (a profundidad grande el suelo está a la media), o flujo nulo $u_x(L)=0$ (se implementa copiando el penúltimo nodo). Son la misma idea desde dos lados; a $L$ grande dan lo mismo.
* **Paso espacial.** $\Delta x = 1$ cm: resuelve $\delta_d \approx 12$ cm con más de 10 nodos, y hace que los bordes de las capas ($7$, $28$, $100$, $255$ cm) caigan en nodos. El costo del esquema crece como $1/\Delta x^3$ (nodos $\propto 1/\Delta x$, pasos $\propto 1/\Delta x^2$ por la condición $r\le1/2$): con $L = 10$ m son 1001 nodos.
* **Paso temporal.** El dato es horario, pero con $\Delta x = 1$ cm y $D = 5\times10^{-7}$ estabilidad exige $\Delta t \le \Delta x^2/(2D) = 100$ s. Se hacen entonces $m = \lceil 3600\,D/(r_{max}\Delta x^2)\rceil$ **sub-pasos** por cada hora del dato, con $r_{max} = 0.4$ (un margen), y el borde se interpola linealmente en el tiempo entre dos datos horarios. Ojo: $m$ crece con $D$; si ajustás un $D$ grande y dejás $\Delta t$ fijo, el esquema explota.
* **Dato inicial.** No conocemos el perfil de temperatura del 1/1 (ni por debajo de $2.55$ m): usamos el **perfil medio**, que interpola las medias anuales de las capas en las profundidades $x_m$ y es constante por debajo (todas las capas tienen una media anual de unos $19$ °C).
* **Spin-up.** Un dato inicial incorrecto contamina la solución hasta que se "olvida": el tiempo característico es $x^2/D$ (unas horas a $x=10$ cm, casi una semana a $x=50$ cm, unos tres meses a $x=2$ m). Se corre entonces antes de la fecha de interés un tramo de **calentamiento** (*spin-up*): acá, un mes, repitiendo cíclicamente el forzado (los 31 días anteriores al 1/1 son los últimos 31 de diciembre). La capa más profunda no llega a olvidar del todo el dato inicial en un mes; un spin-up de un año lo prueba (Tarea 5, punto (c)): cambia la capa profunda en unas décimas de grado, mucho menos que los residuos de (b), así que estos no vienen del dato inicial.

### Qué borde usar

**La capa superficial, no el aire.** El modelo pide la temperatura *del suelo* en $x=0$. La temperatura del aire (a 2 m) no es esa: el suelo se calienta por radiación solar y se enfría por radiación infrarroja y evaporación, y la interfaz aire–suelo transfiere calor con un coeficiente finito, no con una igualdad (una condición de Dirichlet con el aire *supone* que el suelo copia al aire). En los datos: la capa $0$–$7$ cm oscila más que el aire (amplitud diaria de unos $4$ °C contra $2.8$ °C; anual de $8$ °C contra $6.9$ °C) y se calienta más en verano. Usar el aire como borde le quita al suelo un tercio de la amplitud diaria antes de empezar (y lo enfría $1$ °C en promedio), y es un error de *modelado* que ningún $D$ compensa (Tarea 8). La capa $0$–$7$ es la observación más cercana a la superficie, y viene del mismo reanálisis que las otras capas: es un forzado *consistente*. Su defecto: es un promedio sobre $7$ cm, no el valor en $x=0$; por eso la capa $0$–$7$ *simulada* (el promedio de la solución sobre $[0,7]$ cm) va a oscilar menos que el dato, aunque el borde sea exactamente el dato (es el efecto de la Tarea 4, al revés). En la Tarea 8 se puede corregir.

### Cómo comparar

Los datos son promedios de capa, así que hay que comparar **promedios de la simulación sobre cada capa** (la solución numérica en cada nodo $\to$ promedio sobre los nodos de $[a,b]$ con la regla del trapecio). Comparar la simulación en $x=x_m$ con el dato sería repetir el error de la Tarea 4. Para cada capa se calculan el **sesgo** (media de $\text{sim}-\text{dato}$) y la **raíz del error cuadrático medio** (RMSE), y se muestran los residuos mes a mes: si son ruido, no dependen del mes; si el modelo omite algo (humedad, lluvia, la estratificación del suelo), el residuo tiene estructura estacional.

**Qué se espera.** Para las capas intermedias ($7$–$28$ y $28$–$100$ cm) un RMSE de unas pocas décimas de grado; para la capa $0$–$7$, del orden de $1$ °C (por el efecto descrito); para la profunda, mayor, del orden de $1$ °C, con sesgo que cambia de signo con la estación.

### Tarea 5: Simulación explícita con la capa superficial como borde

Completá `simular_suelo(D, Ts, L, dx, spin, fondo, T_capas_ini, T_fondo, r_max)`: resuelve el problema con el esquema explícito y devuelve `(capas, x, u_min)`: `capas` es un array `(len(Ts), 4)` con el promedio de la solución sobre cada capa en cada instante del dato (una fila por hora); `x` la grilla; y `u_min` el mínimo, en cada nodo, de la temperatura durante todo el período comparado (lo usa la Tarea 7). Ya está escrita la parte de preparación (grilla, dato inicial, sub-pasos, ciclo de spin-up y el cálculo de los promedios `promedios_capa`); te falta el bucle temporal:

* un sub-paso del esquema explícito en los nodos interiores (con `r`), *in place* y vectorizado;
* el borde izquierdo, interpolando linealmente entre `Ts[i]` y `Ts[i+1]` en cada sub-paso;
* el borde derecho, según `fondo` (`"T"`: Dirichlet $T_L$; `"F"`: flujo nulo).

**(a) Verificación de la implementación.** La celda de verificación (abajo) simula 30 días con un forzado exactamente periódico, $T_s = 20 + 5\cos(\omega_d t)$, y compara cada capa con el promedio de la solución periódica, $20 + \operatorname{Re}[5\,g(\delta; a, b)\,e^{i\omega_d t}]$ (con la $g$ de la Tarea 4): tiene que pasar antes de seguir.

**(b) La simulación del año.** Con `D_sim` (elegilo: la media geométrica de tus dos $D$ de la Tarea 4, o $5\times10^{-7}$) y `Ts = Y[:, 0]` (la capa superficial como borde) simulá 2023 y calculá el sesgo y el RMSE de cada capa. Graficá (i) una semana de enero: datos y simulación en las cuatro capas; (ii) los residuos (sim $-$ dato) de cada capa en medias diarias durante el año; (iii) el RMSE por mes y por capa (mapa de colores).

**(c) Sensibilidad numérica y de modelado.** Repetí (b) con: $L = 5$ m; flujo nulo en $L = 10$ m; spin-up de un año (`spin=365`); $\Delta x = 2$ cm. Mostrá, para cada variante, la diferencia máxima con la corrida base en cada capa. Verificá también que con $L = 2$ m (menos que el fondo de la última capa) el resultado es absurdo: la función tiene que avisar (`assert`) que $L > 2.55$ m.

**(d) Barrido de $D$.** Corré (b) para $D \in \{3, 4, 5, 6, 8\}\times10^{-7}$ y graficá el RMSE de cada capa contra $D$. ¿Qué $D$ minimiza el error en cada capa?

**Qué se espera.** (a) Diferencias de unos centésimos de grado. (b) Ver arriba. (c) Diferencias de menos de $0.15$ °C en las capas de $0$ a $100$ cm en todas las variantes, y de unos $0.3$ °C como máximo en la capa profunda ($L=5$ m y spin-up de un año); el flujo nulo casi no cambia nada. Los errores de (b) no vienen de estas decisiones numéricas: son del modelo. (d) Las capas intermedias y la profunda tienen su mínimo entre $5$ y $6\times10^{-7}$ (la superficial prefiere un $D$ mayor porque su defecto es otro), cerca de tu estimación anual con promedio de capa y sensiblemente mayor que el $D$ de fase del ciclo anual; el mínimo es plano: entre $4$ y $6\times10^{-7}$ el RMSE cambia menos del $20\,\%$. El $D$ que mejor ajusta la *simulación* es el compromiso entre todo lo que el modelo no captura.

In [ ]:
def promedios_capa(u, x):
    """Temperatura media de cada capa: regla del trapecio acumulada, interpolada en los bordes de las capas."""
    F = np.concatenate([[0.0], np.cumsum(0.5 * (u[1:] + u[:-1]) * np.diff(x))])
    return np.diff(np.interp(bordes, x, F)) / np.diff(bordes)

def simular_suelo(D, Ts, L=10.0, dx=0.01, spin=31, fondo="T", T_capas_ini=None, T_fondo=None, r_max=0.4):
    """Esquema explícito para u_t = D u_xx en (0, L) con u(0, t) = Ts(t) (dato horario, interpolado linealmente).
    Devuelve (capas, x, u_min): promedio por capa en cada hora, grilla y mínimo de u en cada nodo."""
    assert L > bordes[-1], "L debe ser mayor que el fondo de la última capa"
    N_ = len(Ts)
    F = Ts[np.arange(-spin * 24, N_) % N_]                    # forzado: spin-up (últimos días del registro) + registro
    x = np.arange(int(round(L / dx)) + 1) * dx
    T_capas_ini = Y.mean(axis=0) if T_capas_ini is None else np.asarray(T_capas_ini, float)
    T_fondo = T_capas_ini[-1] if T_fondo is None else T_fondo
    u = np.interp(x, x_m, T_capas_ini)                        # dato inicial: perfil medio
    m = int(np.ceil(3600 * D / (r_max * dx ** 2)))            # sub-pasos por hora
    r = D * (3600 / m) / dx ** 2
    assert r <= 0.5
    capas = np.empty((N_, 4)); u_min = np.full(len(x), np.inf)
    for i in range(len(F) - 1):
        if i >= spin * 24:
            capas[i - spin * 24] = promedios_capa(u, x)
            u_min = np.minimum(u_min, u)
        for j in range(1, m + 1):
            # TODO: un sub-paso explícito en los nodos interiores (in place)
            # TODO: borde izquierdo: interpolación lineal entre F[i] y F[i+1] (fracción j/m)
            # TODO: borde derecho según `fondo`
            pass
    capas[-1] = promedios_capa(u, x)
    return capas, x, u_min

D_sim = ...    # TODO
# (b) simulación del año, sesgo, RMSE y figura; (c) sensibilidad; (d) barrido de D

In [ ]:
# Verificación: forzado exactamente periódico, 30 días; la solución tiene que ser el promedio de u_per sobre cada capa
t30 = np.arange(720) * 3600.0
Ts_test = 20 + 5 * np.cos(omega_d * t30)
sim_t = simular_suelo(D_tipico, Ts_test, spin=15, T_capas_ini=[20] * 4)[0]
exacta = np.array([20 + np.real(5 * ganancia(delta_de(D_tipico, omega_d), bordes[i], bordes[i + 1]) * np.exp(1j * omega_d * t30)) for i in range(4)]).T
err_a = np.abs(sim_t - exacta)[-240:].max(axis=0)
print("error máximo de la simulación contra u_per promediada, últimos 10 días, por capa (°C):", np.round(err_a, 3))
assert sim_t.shape == (720, 4) and err_a.max() < 0.1, "la simulación no reproduce la solución periódica"
try:
    simular_suelo(D_tipico, Ts_test[:48], L=2.0); raise RuntimeError("debería haber fallado con L = 2 m")
except AssertionError:
    pass
print("simulación: OK")

## 6. Con un mes de datos: ¿cuánta precisión es razonable?

En la práctica no siempre hay un año de registros: una campaña de medición puede durar semanas. ¿Cuánto cambia $D$ si solo se usa un mes? Con un mes **no se puede estimar el ciclo anual** (el registro es más corto que el período: el ciclo anual se ve como una tendencia), pero sí el diario: en 31 días hay 31 ciclos completos.

**Cómo estimar.** Para cada mes se ajusta a cada capa, por mínimos cuadrados con frecuencia fija (Sección 2), una **tendencia cúbica** más los armónicos $\omega_d$ y $2\omega_d$ (la tendencia absorbe la parte lenta: el ciclo anual y los cambios de tiempo; el armónico de 12 h, que en la capa superficial tiene una amplitud de más de $1$ °C, evita que la forma no sinusoidal del ciclo diario lo contamine). Con la amplitud y la fase de las tres capas superficiales se calculan tres estimaciones de $D$ del ciclo diario: por amplitud y por fase con las dos primeras capas (como en la Tarea 3), y con el ajuste con promedio de capa de las tres (como en la Tarea 4).

**Qué se espera.** Las estimaciones mensuales varían entre meses en un $\pm 10$–$20\,\%$ alrededor de su media ($\approx 4\times10^{-7}$), con una estructura estacional (las tres son mayores de abril a septiembre, otoño e invierno, y menores de octubre a marzo), y el orden entre los métodos casi siempre se mantiene ($D$ por fase mayor que por amplitud). Esa estructura es información, no ruido: sugiere que $D$ **no es constante** (la conductividad térmica del suelo mojado es mucho mayor que la del seco, y depende de la temperatura; es una hipótesis que este laboratorio no prueba). Una campaña de un mes da entonces un $D$ *de ese mes*, con una variación de decenas de por ciento respecto de otro mes y del $D$ "de todo el año". Comparar, para enero y julio, con las estimaciones del año completo.

### Tarea 6: Un mes de verano y uno de invierno (y los demás)

Escribí `D_del_mes(m)`: con las filas del mes `m` (`mes = suelo.fecha_hora.dt.month.values`), ajusta con `ls_ciclos(..., [omega_d, 2*omega_d], grado=3)` las tres capas superficiales, y devuelve las tres estimaciones de $D$: por amplitud (dos capas), por fase (dos capas) y con promedio de capa (tres capas, `ajustar_capas`).

Calculalo para los 12 meses; hacé una tabla y graficá las tres estimaciones contra el mes, marcando enero (verano) y julio (invierno), junto con las líneas horizontales de las estimaciones del año completo de la Tarea 3 (diario) y de $D_{capa}$ de la Tarea 4. Resumí, para cada método, la media y el desvío entre meses.

**Qué se espera.** Valores entre $2\times10^{-7}$ y $6\times10^{-7}$ m$^2$/s: dentro de un factor 2 del valor típico, con dispersión entre meses de $10$–$20\,\%$. El $D$ por fase es mayor que el $D$ por amplitud en casi todos los meses, y el de capa (tres capas) es del mismo orden. Anotá qué mes da el $D$ más bajo y cuál el más alto, y si enero y julio difieren entre sí más que lo que difieren los métodos.

In [ ]:
mes = suelo.fecha_hora.dt.month.values

def D_del_mes(m):
    """Devuelve (D por amplitud, D por fase, D con promedio de capa) del ciclo diario del mes m."""
    # TODO: ls_ciclos con [omega_d, 2*omega_d], grado=3, para las capas 0, 1, 2 -> amplitud y fase de omega_d
    #       D por amplitud y por fase con las capas 0 y 1; ajustar_capas(C, range(3), omega_d)["D"]
    pass

# TODO: 12 meses, tabla, resumen (media y desvío) y figura

In [ ]:
# Verificación
_ene, _jul = D_del_mes(1), D_del_mes(7)
assert all(1e-7 < v < 1e-6 for v in (*_ene, *_jul)), "cada estimación mensual debería estar entre 1e-7 y 1e-6"
assert _ene[1] > _ene[0] and _jul[1] > _jul[0], "por fase debería dar más que por amplitud"
print("un mes de datos: OK")

## 7. ¿A qué profundidad enterrar una cañería? (pregunta 5)

**Con la solución periódica.** Para el ciclo anual, $u_{per} = \bar T + A e^{-x/\delta_a}\cos(\omega_a t - x/\delta_a + \varphi_s)$ y la temperatura **mínima del año** a profundidad $x$ es $\bar T - A e^{-x/\delta_a}$ (el mínimo del coseno). La cañería no se congela si esa mínima es positiva:

$$\bar T - A e^{-x/\delta_a} > 0 \iff x > x^* = \delta_a\ln\frac{A}{\bar T}\qquad(\text{si } A > \bar T;\ \text{si no, no se congela ni en la superficie}).$$

Como $\delta_a = \sqrt{2D/\omega_a}$, **$x^*$ es proporcional a $\sqrt D$**: un error del $50\,\%$ en $D$ es un error del $22\,\%$ en la profundidad. Es la razón por la que la respuesta útil no es "$x^* = 2.4$ m" sino un *rango*, que sale de la incertidumbre de $D$ (Tareas 3, 4 y 6): con las cuatro estimaciones de la Tarea 3, $x^*$ va de unos $2$ m a más de $4$ m para el mismo clima. Los parámetros que entran son la temperatura media $\bar T$ y la amplitud anual $A$ **de la superficie**: usamos $\bar T$ de la capa superficial y $A = A_0$ del ajuste con promedio de capa (Tarea 4).

**Lo que la fórmula no tiene.** (i) El ciclo anual real no es un coseno perfecto: hay olas de frío de días o semanas que bajan la temperatura de la superficie por debajo de $\bar T - A$, y penetran menos (una perturbación de período $\tau$ penetra $\delta = \sqrt{D\tau/\pi}$: unos $0.4$ m para dos semanas). (ii) Los extremos diarios se apagan en los primeros $30$ cm. (iii) El congelamiento libera calor latente y cambia $D$. Por eso conviene *verificar* la fórmula con una simulación (la Tarea 5 con otro forzado) y agregar **un margen**.

**La simulación como verificación.** Para el clima frío hipotético ($\bar T = 5$ °C, $A = 15$ °C) construimos un forzado con el mismo *tiempo* meteorológico del dato, reescalado: $T_s^{frío}(t) = 5 + \frac{15}{A_s}\bigl(T_s(t) - \overline{T_s}\bigr)$ con $A_s$ la amplitud anual de la capa superficial (así conserva los ciclos diarios y las olas de frío, ahora amplificados). Se simula con un *spin-up* de **dos años** (repitiendo el forzado cíclicamente: la solución converge al atractor periódico, que es el estado que describe la solución periódica; el dato inicial se olvida), dato inicial $5$ °C, y se lee el mínimo de $u$ en cada profundidad a lo largo del año, `u_min` (ya lo devuelve `simular_suelo`).

**Qué se espera.** Para Buenos Aires ($\bar T \approx 19$ °C, $A \approx 8.7$ °C): $A < \bar T$, no hay congelamiento, y la mínima anual del modelo periódico es de unos $10$ °C en la superficie (la mínima horaria real de la capa superficial es de $1$ °C: la fórmula no ve los extremos diarios ni las olas de frío; de $64$ cm hacia abajo el mínimo del dato ya está a $1$ °C de la curva). Para el clima frío: $x^* = \delta_a\ln 3 \approx 1.1\,\delta_a$, es decir entre $2$ y $4.4$ m según el $D$ (con el ajuste de la Tarea 4, unos $2.7$ m); la simulación, que tiene los extremos y las olas de frío, da **prácticamente lo mismo que la fórmula con el mismo $D$** (un desvío de centímetros) aunque la mínima en la superficie es mucho más baja: los extremos rápidos se apagan antes de los $50$ cm. Conclusión: la incertidumbre que domina es la de $D$, no la del clima.

### Tarea 7: La cañería, con los valores ajustados y con un clima frío

Escribí `x_libre(T_media, A, delta)`, la profundidad mínima $x^*$ (0 si no hay congelamiento en la superficie).

**(a) Buenos Aires.** Con $\bar T$ = media de la capa superficial, $A = A_0$ (amplitud anual en $x=0$ del ajuste con promedio de capa de la Tarea 4: `fit4[("anual", 4, "capa")]["A0"]`) y $D = D_{capa,a}$, ¿se congela? Calculá la mínima anual del modelo periódico a cada profundidad y compará con el mínimo del dato de cada capa (`Y.min(axis=0)`).

**(b) Clima frío hipotético.** Con $\bar T = 5$ °C y $A = 15$ °C, calculá $x^*$ para cada uno de los $D$ estimados (las cuatro estimaciones de la Tarea 3, `D_capa_a`, `D_capa_d` y `D_tipico`) y armá una tabla con $\delta_a$ y $x^*$.

**(c) Verificación con la simulación.** Con el forzado reescalado `Ts_frio` de arriba y `D = D_sim`, corré `simular_suelo(..., spin=730, T_capas_ini=[5]*4, T_fondo=5)` y sacá de `u_min` la profundidad a la que la mínima del año cruza $0$ °C. Compará con la fórmula (con el mismo $D$) y con el rango de (b). Graficá en dos paneles: (a) Buenos Aires: mínima anual del modelo periódico contra $x$ y los mínimos del dato (marcados en $x_m$); (b) clima frío: $\bar T - A e^{-x/\delta_a}$ para tres valores de $D$ y el `u_min` de la simulación.

**Qué se espera.** Buenos Aires: no se congela (mínima del modelo $\approx 10.6$ °C en la superficie; el mínimo horario real de la capa superficial es mucho menor, de una noche de invierno, y de $64$ cm hacia abajo el mínimo del dato está cerca de la curva). Clima frío: $x^*$ entre $2$ y $4.4$ m según el $D$; la simulación, muy cerca de la fórmula con el mismo $D$.

In [ ]:
def x_libre(T_media, A, delta):
    """Profundidad mínima para que la mínima anual T_media - A exp(-x/delta) sea positiva (0 si no hay congelamiento)."""
    # TODO
    pass

# (a) Buenos Aires; (b) clima frío para cada D; (c) simulación con el forzado reescalado
A_s = tab_dft.loc["0–7 cm", "A_anual"]
Ts_frio = 5 + 15 / A_s * (Y[:, 0] - Y[:, 0].mean())
# TODO: tablas y figura de dos paneles

In [ ]:
# Verificación
assert x_libre(19.0, 8.0, 2.2) == 0.0 and abs(x_libre(5.0, 15.0, 2.24) - 2.24 * np.log(3)) < 1e-12
print("cañería: OK")

## 8. (Opcional) El aire como borde, y un borde corregido

La Sección 5 argumentó que la capa superficial es mejor forzado que el aire. Acá se mide. Se comparan tres bordes con el mismo $D$ y la misma simulación:

1. **el aire** (`t_aire`), la primera aproximación que sugiere el enunciado;
2. **la capa $0$–$7$ cm** (la Tarea 5);
3. **la capa $0$–$7$ cm *corregida* por su promedio**: el dato es $\bar u_{[0,b]}$ ($b = 7$ cm), y lo que necesita el modelo es $u(0,t)$. Por la Tarea 4, cada componente de frecuencia $\omega$ de $u(0,t)$ aparece en el promedio multiplicada por $g(\delta(\omega); 0, b)$ (con $\delta(\omega) = \sqrt{2D/\omega}$); entonces se recupera $u(0,t)$ **dividiendo el espectro de la capa por $g(\delta(\omega);0,b)$** en cada frecuencia (una *deconvolución*, que amplifica las frecuencias altas: a $\omega$ horaria, $|g|\approx 0.4$: es una amplificación moderada y no hay problema de ruido) y antitransformando. Con `imc.espectro.filtrar(x, dt, mascara)` es una línea: `mascara(xi)` devuelve los factores complejos $1/g$, con $\omega = 2\pi\xi$ y el factor $1$ en $\xi = 0$.

**Qué se espera.** El RMSE con el aire como borde es $2$–$3$ veces mayor que con la capa superficial en las capas intermedias (y casi igual en la profunda, donde domina otra falla del modelo); parte del error es un sesgo de $-1$ °C (el suelo es en promedio más caliente que el aire, por la radiación) y el resto es la amplitud diaria y la fase, y ningún $D$ lo compensa. La corrección deja el error de la capa $0$–$7$ cm en unas centésimas de grado (ojo: es casi automático, es la capa con la que se construyó el borde), no cambia las profundas, y **empeora un poco la capa $7$–$28$ cm**: con el borde corregido el modelo homogéneo ya no puede acomodar a la vez a las dos capas superficiales, y el error se traslada. Es una señal más de que el modelo de una sola $D$ no es del todo correcto para este dato.

### Tarea 8: (Opcional) El aire como borde y el borde corregido

Escribí `borde_corregido(y, D, b=0.07)`: aplica `espectro.filtrar` con la máscara $1/g(\delta(\omega); 0, b)$ (con $g$ de la Tarea 4; `mascara` recibe `xi` en Hz y tiene que devolver un array complejo del mismo largo, valiendo 1 en `xi = 0`) y devuelve la serie corregida. Verificá que la media no cambia y que el desvío aumenta entre un $5$ y un $20\,\%$.

Simulá con los tres bordes (aire, capa $0$–$7$, capa corregida) con `D_sim` y calculá el RMSE y el sesgo de cada capa. Graficá el RMSE por capa como barras agrupadas.

**Qué se espera.** Ver arriba. Anotá para cada capa cuál de los tres bordes gana y qué parte del error del aire es sesgo (usá $\text{RMSE}^2 = \text{sesgo}^2 + \text{varianza}$).

In [ ]:
def borde_corregido(y, D, b=0.07):
    """Serie y (promedio de [0, b]) deconvolucionada para recuperar u(0, t): espectro / g(delta(omega); 0, b)."""
    # TODO: def mascara(xi): ... ; return espectro.filtrar(y, 3600.0, mascara)
    pass

# TODO: tres simulaciones (aire, Y[:, 0], borde_corregido(Y[:, 0], D_sim)), RMSE y sesgo por capa, barras agrupadas

In [ ]:
# Verificación
_c = borde_corregido(Y[:, 0], 5e-7)
assert abs(_c.mean() - Y[:, 0].mean()) < 1e-6 and 1.05 < _c.std() / Y[:, 0].std() < 1.25, "la corrección debe conservar la media y aumentar el desvío entre 5 y 25 %"
print("borde corregido: OK")

## Interpretación

Esta es la parte que se evalúa. Respondé en las celdas de texto que siguen, con oraciones completas y citando los números o gráficos que obtuviste.

**1.** **El modelo y el borde (pregunta 1 del problema).** Escribí el problema que simulaste: la ecuación, el dominio, las condiciones de borde y el dato inicial, y justificá cada elección (por qué $L = 10$ m y esa condición inferior; por qué la capa superficial y no el aire, con los errores de las Tareas 5 y 8). Enumerá al menos tres hipótesis del modelo que el suelo real (o el reanálisis) no cumple.

*(escribí tu respuesta acá)*

**2.** **Penetración (pregunta 2).** ¿Por qué el ciclo diario penetra menos que el anual? ¿Cuánto menos, y de qué depende la profundidad de penetración? Con tus $\delta$ de las Tareas 3 y 4 y las amplitudes de la Tarea 2, calculá qué fracción de la amplitud de la superficie llega a $17.5$ cm y a $64$ cm en cada ciclo, y compará con lo que dice la teoría ($e^{-x/\delta}$, cociente $\sqrt{365}$ entre los $\delta$).

*(escribí tu respuesta acá)*

**3.** **Retraso (pregunta 3).** ¿Cuánto vale el retraso a cada profundidad? Con tus estimaciones, dá el retraso del ciclo diario entre las capas $0$–$7$ y $7$–$28$ cm (en horas) y el del anual entre la capa $0$–$7$ y la $100$–$255$ cm (en días), y compará con $x/(\delta\omega)$. ¿Por qué el retraso *de una capa* no es el del punto medio, y cuándo importa (Tarea 4)?

*(escribí tu respuesta acá)*

**4.** **Estimar $D$ y comprobar el modelo (pregunta 4).** Resumí en una tabla tus estimaciones de $D$ (Tareas 3, 4, 5 y 6) con su incertidumbre. ¿Coinciden las cuatro estimaciones de la Tarea 3? ¿Deberían? Explicá por qué no, separando lo que *explica* el promedio de capa (con los números de la Tarea 4) de lo que *no* explica (¿qué queda entonces?), y qué te dicen sobre el modelo los residuos de la simulación (dónde y cuándo falla; humedad, lluvia, estratificación del suelo del reanálisis). Con la variación mes a mes de la Tarea 6, ¿qué precisión es razonable esperar de $D$, y qué valor y qué error informarías?

*(escribí tu respuesta acá)*

**5.** **La cañería (pregunta 5).** ¿A qué profundidad enterrarías la cañería en Buenos Aires y en el clima frío hipotético? Dá un *rango* (no un número) que refleje la incertidumbre de $D$, explicá por qué es proporcional a $\sqrt D$, y decidí qué margen agregarías, argumentando con lo que la simulación de la Tarea 7 tiene y la fórmula no.

*(escribí tu respuesta acá)*